### 1. Imports

In [1]:
import pandas as pd
import glob
import os
from functools import reduce
import numpy as np

## 2. ETL

### 2.1 Estatística Educação Básica

In [2]:
file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/Sinopse_Estatistica_da_Educação_Basica_2024.xlsx'


In [3]:
def etl_educacao_sheet_1_2(caminho_arquivo):
    print("Loading raw Excel sheet 1.2...")
    df_1_2 = pd.read_excel(caminho_arquivo, sheet_name='1.2', skiprows=10, header=None)
    df_1_2.columns = [
        'Região Geográfica', 
        'Unidade da Federação', 
        'Município', 
        'Código do Município', 
        'Total Geral',
        'Urbana - Total',
        'Urbana - Federal',
        'Urbana - Estadual',
        'Urbana - Municipal',
        'Urbana - Privada',
        'Rural - Total',
        'Rural - Federal',
        'Rural - Estadual',
        'Rural - Municipal',
        'Rural - Privada'
    ]
    print("Cleaning metadata rows...")
    df_cleaned = df_1_2.copy()
    
    # Filter by numeric IBGE code to gracefully drop country/regional summaries and footer notes built into the excel
    df_cleaned = df_cleaned[pd.to_numeric(df_cleaned['Código do Município'], errors='coerce').notnull()]
    df_cleaned['Código do Município'] = df_cleaned['Código do Município'].astype(int)
    # Identifiers we want to anchor the dataset on
    id_vars = ['Região Geográfica', 'Unidade da Federação', 'Município', 'Código do Município']
    value_vars = [col for col in df_cleaned.columns if col not in id_vars]
    print("Melting (unpivoting) dataset...")
    df_melted = df_cleaned.melt(id_vars=id_vars, value_vars=value_vars, var_name='Localizacao_Dependencia', value_name='Numero_Escolas')
    print("Extracting dimensions...")
    # Example string manipulation: 'Urbana - Municipal' -> Localizacao: 'Urbana', Dependencia: 'Municipal'
    # Exception handling: 'Total Geral' -> Localizacao: 'Total', Dependencia: 'Total Geral'
    df_melted['Localizacao'] = df_melted['Localizacao_Dependencia'].apply(lambda x: x.split(' - ')[0] if ' - ' in x else 'Total')
    df_melted['Dependencia_Administrativa'] = df_melted['Localizacao_Dependencia'].apply(lambda x: x.split(' - ')[1] if ' - ' in x else 'Total Geral')
    # Now drop the old combined column
    df_melted = df_melted.drop(columns=['Localizacao_Dependencia'])
    # Clean numeric format (replace hyphens, dots with NaN) and filter empty measurements gracefully
    df_melted['Numero_Escolas'] = pd.to_numeric(df_melted['Numero_Escolas'], errors='coerce')
    df_melted = df_melted.dropna(subset=['Numero_Escolas'])
    df_melted['Numero_Escolas'] = df_melted['Numero_Escolas'].astype(int)
    # Reorder and rename everything elegantly
    df_final = df_melted[['Código do Município', 'Região Geográfica', 'Unidade da Federação', 'Município', 'Localizacao', 'Dependencia_Administrativa', 'Numero_Escolas']]
    df_final.columns = ['cod_ibge', 'regiao', 'uf', 'municipio', 'localizacao', 'dependencia', 'num_escolas']
    return df_final

In [4]:
df_educacao_final = etl_educacao_sheet_1_2(file_path)
print(f"\nETL completed. Final shape: {df_educacao_final.shape}")
display(df_educacao_final.head(10))

Loading raw Excel sheet 1.2...
Cleaning metadata rows...
Melting (unpivoting) dataset...
Extracting dimensions...

ETL completed. Final shape: (61270, 7)


,cod_ibge,regiao,uf,municipio,localizacao,dependencia,num_escolas
0,1100015,Norte,Rondônia,Alta Floresta D'Oeste,Total,Total Geral,5071
1,1100379,Norte,Rondônia,Alto Alegre dos Parecis,Total,Total Geral,2569
2,1100403,Norte,Rondônia,Alto Paraíso,Total,Total Geral,3162
3,1100346,Norte,Rondônia,Alvorada D'Oeste,Total,Total Geral,2822
4,1100023,Norte,Rondônia,Ariquemes,Total,Total Geral,23098
5,1100452,Norte,Rondônia,Buritis,Total,Total Geral,6830
6,1100031,Norte,Rondônia,Cabixi,Total,Total Geral,1189
7,1100601,Norte,Rondônia,Cacaulândia,Total,Total Geral,1050
8,1100049,Norte,Rondônia,Cacoal,Total,Total Geral,21071
9,1100700,Norte,Rondônia,Campo Novo de Rondônia,Total,Total Geral,2195


In [5]:
# Save output
output_csv = '/home/raimundoivy/Documents/pad_avaliação_02/dados/educacao_basica_2024_1_2_cleaned.csv'
output_parquet = '/home/raimundoivy/Documents/pad_avaliação_02/dados/educacao_basica_2024_1_2_cleaned.parquet'
print(f"\nSaving to {output_csv} and {output_parquet}...")
df_educacao_final.to_csv(output_csv, index=False, encoding='utf-8')
df_educacao_final.to_parquet(output_parquet, index=False)
print("Done!")


Saving to /home/raimundoivy/Documents/pad_avaliação_02/dados/educacao_basica_2024_1_2_cleaned.csv and /home/raimundoivy/Documents/pad_avaliação_02/dados/educacao_basica_2024_1_2_cleaned.parquet...
Done!


### 2.2 tabela6873 -- Tratores

In [6]:
file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873.csv'

In [7]:
def etl_tabela6873(caminho_arquivo):
    print("Loading raw CSV...")
    df_raw = pd.read_csv(
        caminho_arquivo, 
        sep=';', 
        skiprows=4, 
        engine='python',
        na_values=['-', 'X', '..', '...', ' '],
        on_bad_lines='skip',
        decimal=',' 
    )
    df_raw.columns = ['cod_ibge', 'localidade', 'condicao_produtor', 'cod_filtro', 'grupos_atividade', 'valor']
    df_raw = df_raw.dropna(subset=['cod_ibge'])
    
    # Filter only rows that have numeric IBGE Codes
    df_raw['is_numeric_ibge'] = df_raw['cod_ibge'].astype(str).str.isnumeric()
    valid_rows = df_raw[df_raw['is_numeric_ibge']].copy()
    
    # Halve the dataset (first half: establishments, second half: tractors)
    n_registros = len(valid_rows) // 2
    
    df_estab = valid_rows.iloc[:n_registros].copy()
    df_maq = valid_rows.iloc[n_registros:].copy()
    
    print("Processing numeric values...")
    df_estab['num_estabelecimentos'] = pd.to_numeric(df_estab['valor'], errors='coerce')
    df_maq['num_tratores'] = pd.to_numeric(df_maq['valor'], errors='coerce')
    
    # We must merge using all identifying composite keys to guarantee a 1:1 match
    merge_keys = ['cod_ibge', 'localidade', 'condicao_produtor', 'grupos_atividade', 'cod_filtro']
    df_maq_slim = df_maq[merge_keys + ['num_tratores']]
    
    print("Merging variable blocks...")
    df_final = pd.merge(
        df_estab,
        df_maq_slim,
        on=merge_keys,
        how='left'
    )
    
    # Extract UF and clean Municipio from localidade name
    df_final['uf'] = df_final['localidade'].str.extract(r'\((.*?)\)$')[0]
    df_final['municipio'] = df_final['localidade'].str.replace(r'\s*\(.*?\)$', '', regex=True)
    
    # Calculate Geo Level (e.g. 1=Brazil, 2=State, 7=City)
    df_final['geo_level'] = df_final['cod_ibge'].astype(str).str.len()
    
    # Drop unneeded identifier columns resulting from merge, AND filter out your requested drops
    cols_to_drop = [
        'condicao_produtor', 'grupos_atividade', 'geo_level', 
        'cod_filtro', 'valor', 'is_numeric_ibge', 'localidade'
    ]
    df_final = df_final.drop(columns=[c for c in cols_to_drop if c in df_final.columns], errors='ignore')
    
    # Reorder remaining analytical metrics neatly
    expected_cols = ['cod_ibge', 'uf', 'municipio', 'num_estabelecimentos', 'num_tratores']
    df_final = df_final[[c for c in expected_cols if c in df_final.columns]]
    
    # Finish enforcing correct integer type for cod_ibge
    df_final['cod_ibge'] = df_final['cod_ibge'].astype(int)
    
    return df_final

In [8]:
df_6873_final = etl_tabela6873(file_path)
print(f"\nETL completed. Final shape: {df_6873_final.shape}")
display(df_6873_final.head())

Loading raw CSV...
Processing numeric values...
Merging variable blocks...

ETL completed. Final shape: (5569, 5)


,cod_ibge,uf,municipio,num_estabelecimentos,num_tratores
0,1,NaN,Brasil,734280.0,1229907.0
1,1,NaN,Norte,35092.0,58436.0
2,2,NaN,Nordeste,53284.0,83866.0
3,3,NaN,Sudeste,208791.0,373952.0
4,4,NaN,Sul,347476.0,517042.0


In [9]:
# Save to output folder
output_csv = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873_cleaned.csv'
output_parquet = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873_cleaned.parquet'
print(f"\nSaving to {output_csv} and {output_parquet}...")
df_6873_final.to_csv(output_csv, index=False, encoding='utf-8')
df_6873_final.to_parquet(output_parquet, index=False)
print("Done!")


Saving to /home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873_cleaned.csv and /home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873_cleaned.parquet...
Done!


### 2.3 tabela6873 -- Área em Hectares

In [10]:
file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773.csv'

In [11]:
def etl_tabela6773(caminho_arquivo):
    print("Loading raw CSV...")
    # Load the raw file, skipping the first 4 metadata rows
    df_raw = pd.read_csv(
        caminho_arquivo, 
        sep=';', 
        skiprows=4, 
        engine='python',
        na_values=['-', 'X', '..', '...', ' '],
        on_bad_lines='skip',
        decimal=',' 
    )
    # Rename columns to standard internal names
    df_raw.columns = ['cod_ibge', 'localidade', 'finalidade', 'renda', 'cod_filtro', 'associacao', 'valor']
    # The dataset has stacked variables: first half is 'Number of Establishments', second half is 'Area in Hectares'
    # Clean out the empty footer notes
    df_raw = df_raw.dropna(subset=['cod_ibge'])
    
    # Filter only rows representing valid IBGE Codes
    df_raw['is_numeric_ibge'] = df_raw['cod_ibge'].astype(str).str.isnumeric()
    valid_rows = df_raw[df_raw['is_numeric_ibge']].copy()
    
    # We halve the valid rows: first half = establishments, second half = area
    n_registros = len(valid_rows) // 2
    
    df_estab = valid_rows.iloc[:n_registros].copy()
    df_area = valid_rows.iloc[n_registros:].copy()
    
    print("Processing columns...")
    df_estab['num_estabelecimentos'] = pd.to_numeric(df_estab['valor'], errors='coerce')
    df_area['area_hectares'] = pd.to_numeric(df_area['valor'], errors='coerce')
    
    # Extract UF and clean Municipio from 'localidade'
    df_estab['uf'] = df_estab['localidade'].str.extract(r'\((.*?)\)$')[0]
    df_estab['municipio'] = df_estab['localidade'].str.replace(r'\s*\(.*?\)$', '', regex=True)
    
    # Calculate Geo Level for possible hierarchical filtering (1 = Brazil, 7 = City)
    df_estab['geo_level'] = df_estab['cod_ibge'].astype(str).str.len()
    
    # Drop intermediate columns that are now redundant
    df_estab = df_estab.drop(columns=['localidade', 'valor', 'is_numeric_ibge'])
    df_area = df_area[['cod_ibge', 'area_hectares']]
    
    print("Merging variable blocks...")
    # Re-assemble the two variables as side-by-side columns
    df_final = pd.merge(
        df_estab,
        df_area,
        on='cod_ibge',
        how='left'
    )
    
    # Reorder the final dataframe nicely
    df_final = df_final[['cod_ibge', 'geo_level', 'uf', 'municipio', 'finalidade', 'renda', 'associacao', 'num_estabelecimentos', 'area_hectares']]
    
    # Adjust types
    df_final['cod_ibge'] = df_final['cod_ibge'].astype(int)
    
    return df_final

In [12]:
df_6773_final = etl_tabela6773(file_path)
print(f"ETL completed. Final shape: {df_6773_final.shape}")
display(df_6773_final.head(30))

Loading raw CSV...
Processing columns...
Merging variable blocks...
ETL completed. Final shape: (5564, 9)


,cod_ibge,geo_level,uf,municipio,finalidade,renda,associacao,num_estabelecimentos,area_hectares
0,1,1,NaN,Brasil,Total,Total,Total,5073324,351289816.0
1,1100015,7,RO,Alta Floresta D'Oeste,Total,Total,Total,2886,372746.0
2,1100023,7,RO,Ariquemes,Total,Total,Total,2928,334256.0
3,1100031,7,RO,Cabixi,Total,Total,Total,1075,113085.0
4,1100049,7,RO,Cacoal,Total,Total,Total,3814,221390.0
5,1100056,7,RO,Cerejeiras,Total,Total,Total,719,126686.0
6,1100064,7,RO,Colorado do Oeste,Total,Total,Total,1674,139796.0
7,1100072,7,RO,Corumbiara,Total,Total,Total,1489,273086.0
8,1100080,7,RO,Costa Marques,Total,Total,Total,1500,220177.0
9,1100098,7,RO,Espigão D'Oeste,Total,Total,Total,2000,247331.0


In [13]:
# Output formats
output_csv = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773_cleaned.csv'
output_parquet = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773_cleaned.parquet'
print(f"\nSaving to {output_csv} and {output_parquet}...")
df_6773_final.to_csv(output_csv, index=False, encoding='utf-8')
df_6773_final.to_parquet(output_parquet, index=False)
print("Done!")


Saving to /home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773_cleaned.csv and /home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773_cleaned.parquet...
Done!


### 2.4 tabela6873 -- Produtos / Área Colhida

In [14]:
file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612.csv'

In [15]:
# Generate the correct column names
years = range(1974, 2025)
products = [
    'Algodão herbáceo (em caroço)',
    'Cana-de-açúcar',
    'Milho (em grão)',
    'Soja (em grão)'
]
columns = ['cod_ibge', 'localidade']
for year in years:
    for prod in products:
        columns.append(f"{year}_{prod}")
print(f"Expected {len(columns)} columns.")

Expected 206 columns.


In [16]:
# Try reading the first line to see how many columns it actually has
with open(file_path, 'r', encoding='utf-8') as f:
    for _ in range(5):
        next(f)
    first_data_line = f.readline().strip()
num_cols_in_data = len(first_data_line.split(';'))
print(f"Data rows seem to have {num_cols_in_data} columns.")

Data rows seem to have 206 columns.


In [17]:
# If there's an extra trailing semicolon, we might read an extra empty column
usecols = list(range(len(columns)))

In [18]:
# Fast C engine
df = pd.read_csv(
    file_path, 
    sep=';', 
    skiprows=5,
    names=columns,
    usecols=usecols,
    na_values=['-', 'X', '..', '...', ' '],
    decimal=',',
    engine='c', # Use C engine for speed
    low_memory=False
)
print(f"Loaded CSV shape: {df.shape}")

Loaded CSV shape: (11167, 206)


In [19]:
# Drop rows that are just notes at the bottom of the CSV (cod_ibge should be numeric)
df = df[pd.to_numeric(df['cod_ibge'], errors='coerce').notnull()]
df['cod_ibge'] = df['cod_ibge'].astype(int)

In [20]:
# Identify levels (Brazil=1, Region=1, State=2, City=7 usually)
df['cod_str'] = df['cod_ibge'].astype(str)
df['geo_level'] = df['cod_str'].str.len()

In [21]:
# Melt the dataframe so it's long format: cod_ibge, localidade, geo_level, year, product, area_colhida
id_vars = ['cod_ibge', 'localidade', 'geo_level']
value_vars = [col for col in columns if col not in ['cod_ibge', 'localidade']]
print("Melting dataframe...")
df_melted = df.melt(id_vars=id_vars, value_vars=value_vars, var_name='year_product', value_name='area_colhida_hectares')
print("Extracting year and product...")
df_melted[['ano', 'produto']] = df_melted['year_product'].str.extract(r'^(\d{4})_(.*)$')
df_melted.drop(columns=['year_product'], inplace=True)
df_melted['ano'] = df_melted['ano'].astype(int)
df_melted['area_colhida_hectares'] = pd.to_numeric(df_melted['area_colhida_hectares'], errors='coerce')
df_melted.dropna(subset=['area_colhida_hectares'], inplace=True)

Melting dataframe...
Extracting year and product...


In [22]:
# Clean localidade name
df_melted['uf'] = df_melted['localidade'].str.extract(r'\((.*?)\)$')[0]
df_melted['municipio'] = df_melted['localidade'].str.replace(r'\s*\(.*?\)$', '', regex=True)

In [23]:
# Reorder columns
df_final = df_melted[['cod_ibge', 'geo_level', 'uf', 'municipio', 'ano', 'produto', 'area_colhida_hectares']]
print(f"ETL completed. Final shape: {df_final.shape}")
display(df_final.head(30))

ETL completed. Final shape: (1065021, 7)


,cod_ibge,geo_level,uf,municipio,ano,produto,area_colhida_hectares
0,1,1,NaN,Brasil,1974,Algodão herbáceo (em caroço),1726036.0
1,1,1,NaN,Norte,1974,Algodão herbáceo (em caroço),256.0
2,2,1,NaN,Nordeste,1974,Algodão herbáceo (em caroço),809101.0
3,3,1,NaN,Sudeste,1974,Algodão herbáceo (em caroço),496122.0
4,4,1,NaN,Sul,1974,Algodão herbáceo (em caroço),310000.0
5,5,1,NaN,Centro-Oeste,1974,Algodão herbáceo (em caroço),110557.0
22,1100205,7,RO,Porto Velho,1974,Algodão herbáceo (em caroço),200.0
168,1500909,7,PA,Augusto Corrêa,1974,Algodão herbáceo (em caroço),9.0
180,1501709,7,PA,Bragança,1974,Algodão herbáceo (em caroço),27.0
190,1502202,7,PA,Capanema,1974,Algodão herbáceo (em caroço),12.0


In [24]:
# Save to a new CSV and parquet
output_csv = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612_cleaned.csv'
output_parquet = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612_cleaned.parquet'
print(f"Saving to {output_csv} and {output_parquet}...")
df_final.to_csv(output_csv, index=False, encoding='utf-8')
df_final.to_parquet(output_parquet, index=False)
print("Done!")

Saving to /home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612_cleaned.csv and /home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612_cleaned.parquet...
Done!


### 2.4 IBGE População

In [25]:
file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio.csv'

In [26]:
def etl_populacao(caminho):
    print("Loading population data...")
    df = pd.read_csv(caminho, sep=',')
    
    print(f"Original shape: {df.shape}")
    print("Years available:", sorted(df['ano'].unique()))
    
    # Cleaning up datatypes to avoid object/float casting issues for downstream JOINS
    df['id_municipio'] = pd.to_numeric(df['id_municipio'], errors='coerce')
    df = df.dropna(subset=['id_municipio'])
    df['id_municipio'] = df['id_municipio'].astype(int)
    
    # Rename keys to universally match the rest of our pipelines
    df = df.rename(columns={'id_municipio': 'cod_ibge', 'sigla_uf': 'uf'})
    
    return df

In [27]:
df_pop_final = etl_populacao(file_path)
print(f"\nETL completed. Final shape: {df_pop_final.shape}")
display(df_pop_final.head(10))

Loading population data...
Original shape: (191099, 4)
Years available: [np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

ETL completed. Final shape: (191098, 4)


,ano,uf,cod_ibge,populacao
0,1991,RO,1100015,31981.0
1,1991,RO,1100023,83684.0
2,1991,RO,1100031,8173.0
3,1991,RO,1100049,79036.0
4,1991,RO,1100056,21607.0
5,1991,RO,1100064,38994.0
6,1991,RO,1100080,10375.0
7,1991,RO,1100098,23157.0
8,1991,RO,1100106,32583.0
9,1991,RO,1100114,63535.0


In [28]:
# Save output
output_csv = '/home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio_cleaned.csv'
output_parquet = '/home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio_cleaned.parquet'
print(f"\nSaving to {output_csv} and {output_parquet}...")
df_pop_final.to_csv(output_csv, index=False, encoding='utf-8')
df_pop_final.to_parquet(output_parquet, index=False)
print("Done!")


Saving to /home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio_cleaned.csv and /home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio_cleaned.parquet...
Done!


## 3. Cruzando os Dados

In [29]:
# Load all the cleaned parquets we built!
print("Loading individual datasets...")
df_pop = pd.read_parquet('/home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio_cleaned.parquet')
df_edu = pd.read_parquet('/home/raimundoivy/Documents/pad_avaliação_02/dados/educacao_basica_2024_1_2_cleaned.parquet')
df_1612 = pd.read_parquet('/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612_cleaned.parquet')
df_6773 = pd.read_parquet('/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773_cleaned.parquet')
df_6873 = pd.read_parquet('/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873_cleaned.parquet')
print("Applying cross-reference logic via 'cod_ibge'...")

Loading individual datasets...
Applying cross-reference logic via 'cod_ibge'...


In [30]:
# 1. Base Table: We use the Population 2024 metric as an anchor since it validates all actual 5,570 active IBGE codes
base = df_pop[df_pop['ano'] == 2024][['cod_ibge', 'uf', 'populacao']].copy()
base = base.rename(columns={'populacao': 'populacao_2024'})

In [31]:
# 2. Add Education Data
# The dataset has multiple rows per localizacao/dependencia, so we safely group purely by IBGE code to get total schools within each municipal border
edu_agg = df_edu.groupby('cod_ibge')['num_escolas'].sum().reset_index()
base = pd.merge(base, edu_agg, on='cod_ibge', how='left')

In [32]:
# 3. Add Agricultural Area (Tabela 1612 - Historical)
# Let's filter for the most recent statistical year recorded (2024 or 2023 depending on crop)
max_ano = df_1612['ano'].max()
df_1612_recent = df_1612[df_1612['ano'] == max_ano]

In [33]:
# Pivot the agricultural products dynamically so we have one row per municipality, and distinct products acting as simple width-columns
agri_pivot = df_1612_recent.pivot_table(
    index='cod_ibge', 
    columns='produto', 
    values='area_colhida_hectares', 
    aggfunc='sum'
).reset_index()

In [34]:
# Rename header strings to snake_case avoiding spaces
agri_pivot.columns = ['cod_ibge'] + [f"area_{c.split(' (')[0].replace('-', '_').replace(' ', '_').lower()}_{max_ano}" for c in agri_pivot.columns if c != 'cod_ibge']
base = pd.merge(base, agri_pivot, on='cod_ibge', how='left')

In [35]:
# 4. Add Agricultural Establishments Area (Tabela 6773)
# To avoid double-matching the State/Region headers, let's strictly filter geo_level to just '7' (City level)
if 'geo_level' in df_6773.columns:
    df_6773_mun = df_6773[df_6773['geo_level'] == 7].drop_duplicates(subset=['cod_ibge'])
else:
    df_6773_mun = df_6773.drop_duplicates(subset=['cod_ibge'])
df_6773_mun = df_6773_mun[['cod_ibge', 'num_estabelecimentos', 'area_hectares']]
df_6773_mun.columns = ['cod_ibge', 'estabelecimentos_agro_total', 'area_agro_total_hectares']
base = pd.merge(base, df_6773_mun, on='cod_ibge', how='left')

In [36]:
# 5. Add Tractors Usage Data (Tabela 6873)
df_6873_mun = df_6873.drop_duplicates(subset=['cod_ibge'])
df_6873_mun = df_6873_mun[['cod_ibge', 'num_tratores']]
base = pd.merge(base, df_6873_mun, on='cod_ibge', how='left')
print(f"\nFinal Unified Data shape (1 row per municipality): {base.shape}")
display(base.head(30))


Final Unified Data shape (1 row per municipality): (5570, 11)


,cod_ibge,uf,populacao_2024,num_escolas,area_algodão_herbáceo_2024,area_cana_de_açúcar_2024,area_milho_2024,area_soja_2024,estabelecimentos_agro_total,area_agro_total_hectares,num_tratores
0,1100015,RO,22853.0,15213,NaN,84.0,9664.0,11758.0,2886.0,372746.0,517.0
1,1100023,RO,108573.0,69294,NaN,730.0,37593.0,109520.0,2928.0,334256.0,501.0
2,1100031,RO,5690.0,3567,NaN,92.0,104069.0,224895.0,1075.0,113085.0,247.0
3,1100049,RO,97637.0,63213,NaN,673.0,16302.0,43797.0,3814.0,221390.0,465.0
4,1100056,RO,16975.0,11127,NaN,616.0,129356.0,265911.0,719.0,126686.0,303.0
5,1100064,RO,16588.0,11121,NaN,59.0,8133.0,15653.0,1674.0,139796.0,248.0
6,1100072,RO,8001.0,4689,NaN,209.0,161976.0,356007.0,1489.0,273086.0,306.0
7,1100080,RO,13522.0,9984,NaN,105.0,19549.0,48038.0,1500.0,220177.0,269.0
8,1100098,RO,32717.0,19809,NaN,NaN,6720.0,12593.0,2000.0,247331.0,334.0
9,1100106,RO,43553.0,33231,NaN,25.0,10998.0,63630.0,602.0,70487.0,91.0


In [37]:
# Save Master Reference dataset
output_master_csv = '/home/raimundoivy/Documents/pad_avaliação_02/dados/master_municipios_agrupado.csv'
output_master_parquet = '/home/raimundoivy/Documents/pad_avaliação_02/dados/master_municipios_agrupado.parquet'
print(f"\nSaving to {output_master_csv} ...")
base.to_csv(output_master_csv, index=False, encoding='utf-8')
base.to_parquet(output_master_parquet, index=False)
print("Complete!")


Saving to /home/raimundoivy/Documents/pad_avaliação_02/dados/master_municipios_agrupado.csv ...
Complete!
